In [3]:
import pandas as pd

# Load all crops
tomato = pd.read_csv("tamato.csv")
potato = pd.read_csv("potato.csv")
onion = pd.read_csv("onion.csv")

# Add crop column
tomato["Crop"] = "Tomato"
potato["Crop"] = "Potato"
onion["Crop"] = "Onion"

# Combine
df = pd.concat([tomato, potato, onion], ignore_index=True)

# Standardize column names
df.rename(columns={
    "Price Date": "date",
    "Modal Price (Rs./Quintal)": "modal_price"
}, inplace=True)

# Parse dates
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Drop NaNs in date or price
df = df.dropna(subset=["date", "modal_price"])

# Save cleaned master file
df.to_csv("cleaned_master.csv", index=False)


C:\Users\darsh\AppData\Local\Temp\ipykernel_19532\1216739226.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["date"], errors="coerce")


In [5]:
import joblib
from sklearn.ensemble import RandomForestRegressor

def train_and_save_model(crop_name):
    df = pd.read_csv("cleaned_master.csv", parse_dates=["date"])
    df = df[df["Crop"] == crop_name].sort_values("date")
    
    # Feature engineering
    for lag in [1, 2, 3]:
        df[f"lag_{lag}"] = df["modal_price"].shift(lag)
    df["ma_3"] = df["modal_price"].rolling(window=3).mean()
    df["ma_7"] = df["modal_price"].rolling(window=7).mean()
    df = df.dropna()

    X = df[["lag_1", "lag_2", "lag_3", "ma_3", "ma_7"]]
    y = df["modal_price"]

    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X, y)

    joblib.dump(model, f"{crop_name.lower()}_model.pkl")

for crop in ["Tomato", "Potato", "Onion"]:
    train_and_save_model(crop)
